# 10 · Test of H7: are our policies under-trained?

**Observation (after notebook 09).** Every run trains for the same 20k steps (batch 32). The official dataset has ~4.5k frames, so 20k steps are ~140 passes over it; `ours_consistent_clean` has ~55k frames: ~12 passes. The training loss of our runs was still falling steadily at 20k (L1 0.127 -> 0.116 over the last 4k steps), larger datasets ended with a higher loss at the same step, and the DART data (whose targets are harder to fit) ended at twice the loss (0.24). An under-trained policy would explain why more data helped only a little (59% -> 67%) and DART not at all.

**Experiment.** Train on the SAME datasets as notebook 09 (pulled from the Hub, no generation) for 60k steps instead of 20k, with checkpoints at 20k, 40k and 60k, and evaluate each twice (n = 100). Learning rate and everything else unchanged. Run once per `VARIANT` (`clean`, `dart`), in two runtimes in parallel. The 20k checkpoint repeats notebook 09's setting (a consistency check).

**Predictions, fixed before running:**
* H7 confirmed: `clean` at 60k >= `clean` at 20k + 10 points.
* H7 rejected: within 5 points.
* H4 (corrective data), re-tested at the longest training: `dart` 60k >= `clean` 60k + 15 points.

**Runtime:** T4, about 5 h per variant (training ~3.8 h, seven evaluations ~1 h); pulls ~1 GB.

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
# LIBERO needs egl_probe / hf-egl-probe, which are built from source with cmake. pip's isolated build cannot run
# the pip-installed cmake wrapper, and the old egl_probe CMakeLists is rejected by cmake 4: build both first,
# without isolation and with the CMake 3.5 policy floor.
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "setuptools", "wheel", "cmake>=3.29,<4.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "egl_probe", "hf-egl-probe"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"],
                   capture_output=True, text=True)
if r.returncode != 0:                                  # show pip's own reason, not just "exit status 1"
    print((r.stdout + r.stderr)[-4000:]); raise RuntimeError("pip install failed: see the lines above")
# Colab preinstalls an older torchao; a newer diffusers (pulled in by lerobot) then fails on import
# (FqnToConfig). ACT does not use torchao, and diffusers skips it when it is absent.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
from p2p.sim.env import ensure_nvidia_egl
ensure_nvidia_egl()      # Colab: make EGL render on the GPU (otherwise it silently renders on the CPU, ~10x slower)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
VARIANT = "clean"          # "clean" or "dart": run this notebook twice, once with each
HUB_REPO, SEED, STEPS, BATCH, EVERY = "phone2panda-data", 0, 60000, 32, 20000
NAME = f"ours_consistent_{VARIANT}"
RUN = f"act_{NAME}_long_s{SEED}"
from p2p.hub import pull, push
pull(HUB_REPO, f"{BASE}/hub", allow_patterns=[f"lerobot/{NAME}/*"])

In [ ]:
!python -m p2p.policy.act train {BASE}/hub/lerobot/{NAME} {BASE}/runs/{RUN} --steps {STEPS} --batch-size {BATCH} --seed {SEED} --save-every {EVERY}

## Evaluate the 20k, 40k and 60k checkpoints, twice each; temporal ensembling on the last

In [ ]:
ckpts = {"20k": "step_020000", "40k": "step_040000", "60k": "final"}
for tag, ck in ckpts.items():
    for k in ("a", "b"):
        nv = 3 if (tag == "60k" and k == "a") else 0
        !python -m p2p.policy.act eval {BASE}/runs/{RUN}/{ck} --episodes 50 --videos {nv} --out {BASE}/runs/{RUN}/eval_{tag}_{k}
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 0 --temporal-ensemble 0.01 --out {BASE}/runs/{RUN}/eval_60k_te
import glob
evals = " ".join(sorted(glob.glob(f"{BASE}/runs/{RUN}/eval*/eval.json")))
!python tools/policy_failures.py {evals}

In [ ]:
push(f"{BASE}/runs/{RUN}", HUB_REPO, path_in_repo=f"runs/{RUN}", ignore_patterns=["step_*/*"])
!cd {BASE}/runs && zip -qr {BASE}/{RUN}.zip {RUN} -x "*/final/*" "*/step_*"
try:
    from google.colab import files; files.download(f"{BASE}/{RUN}.zip")
except Exception:
    print("download", f"{BASE}/{RUN}.zip", "from the Output panel")